# GemCol Evaluation: Phase 6 (Ablation Studies)
This notebook contains ablation experiments to justify architectural choices made in the TriRank pipeline, specifically:
- Candidate pool size variation (Top-K before reranking)
- RRF `k` smoothing parameter sensitivity
- Parallel vs Sequential execution latency
- Weaker embedding baseline (MiniLM-L6-v2) to prove BGE's necessity

In [ ]:
import sys
import os
import json
import time
import torch
import torch.nn as nn
from tqdm import tqdm
import concurrent.futures
from transformers import BertPreTrainedModel, BertModel, AutoTokenizer, BertConfig
from huggingface_hub import hf_hub_download
import safetensors.torch

sys.path.insert(0, '/workspace/gemcol_evaluation')
from utils import load_msmarco_dev, rrf_fusion, load_run_json, evaluate_run, print_metrics, LatencyProfiler, ExperimentTracker

print("Loading dataset...")
queries, qrels, corpus = load_msmarco_dev()

print("Loading base runs...")
bm25_run = load_run_json("/workspace/results/bm25_run.json")
bge_run = load_run_json("/workspace/results/bge_run.json")


## 1. Setup ColBERTv2

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

class ColBERT(nn.Module):
    def __init__(self):
        super().__init__()
        config = BertConfig.from_pretrained("colbert-ir/colbertv2.0")
        self.bert = BertModel(config)
        self.linear = nn.Linear(config.hidden_size, 128, bias=False)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        embeddings = self.linear(outputs.last_hidden_state)
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=-1)
        return embeddings

print("Loading ColBERTv2 model natively...")
tokenizer = AutoTokenizer.from_pretrained("colbert-ir/colbertv2.0")
colbert_model = ColBERT()
model_path = hf_hub_download("colbert-ir/colbertv2.0", "model.safetensors")
state_dict = safetensors.torch.load_file(model_path)
colbert_model.load_state_dict(state_dict, strict=False)
colbert_model = colbert_model.to(device)
colbert_model.eval()
print("✅ Model loaded!")

def encode_query(query_text):
    q_text = "[unused0] " + query_text
    inputs = tokenizer(q_text, return_tensors="pt", max_length=32, truncation=True, padding="max_length").to(device)
    inputs['input_ids'][inputs['input_ids'] == tokenizer.pad_token_id] = tokenizer.mask_token_id
    with torch.no_grad():
        Q = colbert_model(**inputs)
    return Q

def encode_documents_and_score(Q, doc_texts):
    d_texts = ["[unused1] " + d for d in doc_texts]
    inputs = tokenizer(d_texts, return_tensors="pt", max_length=150, truncation=True, padding=True).to(device)
    with torch.no_grad():
        D = colbert_model(**inputs)
    sim = torch.einsum("iqd,bjd->bqj", Q, D)
    d_mask = inputs['attention_mask'].unsqueeze(1)
    sim = sim.masked_fill(d_mask == 0, -10000.0)
    max_scores, _ = torch.max(sim, dim=2)
    final_scores = torch.sum(max_scores, dim=1)
    return final_scores

## 2. Ablation: Candidate Pool Size Variation
Varying `top_k` from 25 to 200 before ColBERT reranking.

In [ ]:
pool_sizes = [25, 50, 100, 200]
pool_metrics = {}
tracker = ExperimentTracker("/workspace/results/experiments.json")

for pool_size in pool_sizes:
    print(f"\n--- Testing Candidate Pool Size: {pool_size} ---")
    fused_run = rrf_fusion([bm25_run, bge_run], k=60, top_k=pool_size)
    
    colbert_run = {}
    prof = LatencyProfiler()
    query_ids = list(fused_run.keys())
    
    for qid in tqdm(query_ids, desc=f"ColBERT Reranking (Pool={pool_size})"):
        query_text = queries[qid]
        candidate_doc_ids = fused_run[qid]
        candidate_texts = [corpus[doc_id] for doc_id in candidate_doc_ids]
        
        with prof.time("colbert_reranking"):
            Q = encode_query(query_text)
            scores = encode_documents_and_score(Q, candidate_texts)
        
        scores = scores.cpu().numpy()
        sorted_pairs = sorted(zip(candidate_doc_ids, scores), key=lambda x: x[1], reverse=True)
        colbert_run[qid] = [doc_id for doc_id, score in sorted_pairs]
        
    metrics = evaluate_run(colbert_run, qrels)
    pool_metrics[pool_size] = metrics
    
    mean_latency = prof.summary()["colbert_reranking"]["mean_ms"]
    print(f"Pool {pool_size} -> nDCG@10: {metrics['ndcg@10']:.4f}, Latency: {mean_latency:.2f} ms")
    
    tracker.log(f"Ablation_PoolSize_{pool_size}", config={"pool_size": pool_size}, metrics={
        "ndcg@10": metrics["ndcg@10"],
        "mrr@10": metrics["mrr@10"],
        "colbert_latency_ms": mean_latency
    })

## 3. Ablation: RRF `k` Parameter Sensitivity
Varying the smoothing parameter `k` in RRF. Standard literature uses 60. We evaluate `k` = [10, 30, 60, 100].

In [ ]:
k_params = [10, 30, 60, 100]

for k in k_params:
    print(f"\n--- Testing RRF k={k} ---")
    fused_run = rrf_fusion([bm25_run, bge_run], k=k, top_k=100)
    metrics = evaluate_run(fused_run, qrels)
    print(f"RRF k={k} -> nDCG@10: {metrics['ndcg@10']:.4f}")
    
    tracker.log(f"Ablation_RRF_k_{k}", config={"rrf_k": k}, metrics={
        "ndcg@10": metrics["ndcg@10"],
        "mrr@10": metrics["mrr@10"]
    })

## 4. Ablation: Parallel vs Sequential Retrieval Latency
Using threading to measure how much time is saved by fetching from BM25 and BGE in parallel.

In [ ]:
# Note: To accurately measure this, we simulate the previously profiled MS MARCO latencies:
# BM25 average: ~60ms
# BGE average: ~300ms
def mock_bm25_query():
    time.sleep(0.060)
    return True

def mock_bge_query():
    time.sleep(0.300)
    return True

print("Running Sequential Retrieval (BM25 then BGE)...")
start = time.time()
mock_bm25_query()
mock_bge_query()
seq_time = time.time() - start
print(f"Sequential Time: {seq_time*1000:.2f} ms")

print("\nRunning Parallel Retrieval...")
start = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
    future_bm25 = executor.submit(mock_bm25_query)
    future_bge = executor.submit(mock_bge_query)
    concurrent.futures.wait([future_bm25, future_bge])
par_time = time.time() - start
print(f"Parallel Time: {par_time*1000:.2f} ms")
print(f"Time Saved: {(seq_time - par_time)*1000:.2f} ms per query")

## 5. Ablation: Weaker Dense Embeddings (MiniLM)
To prove that BGE-large-en-v1.5 is critical, we substitute it with `sentence-transformers/all-MiniLM-L6-v2` (384 dims).
This cell will encode the corpus using MiniLM. *Warning: this will take hours to run.*

In [ ]:
# Uncomment and run to compute the MiniLM exact dense baseline
"""
from sentence_transformers import SentenceTransformer
import math

print("Loading MiniLM model...")
minilm = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)

# Encode Queries
print("Encoding queries...")
query_ids = list(queries.keys())
query_texts = [queries[qid] for qid in query_ids]
query_embeddings = minilm.encode(query_texts, convert_to_tensor=True, batch_size=256)

# Chunked Exact Search against Corpus
chunk_size = 50000
num_chunks = math.ceil(len(corpus) / chunk_size)
doc_ids = list(corpus.keys())

# Track Top 100
top_k_scores = torch.full((len(queries), 100), -float('inf'), device=device)
top_k_indices = torch.zeros((len(queries), 100), dtype=torch.long, device=device)

for i in tqdm(range(num_chunks), desc="MiniLM Exact Search Chunks"):
    start_idx = i * chunk_size
    end_idx = min((i + 1) * chunk_size, len(corpus))
    
    chunk_doc_ids = doc_ids[start_idx:end_idx]
    chunk_texts = [corpus[doc_id] for doc_id in chunk_doc_ids]
    
    chunk_embeddings = minilm.encode(chunk_texts, convert_to_tensor=True, batch_size=512)
    
    # Cosine Similarity
    similarities = torch.matmul(query_embeddings, chunk_embeddings.T)
    
    # Combine with previous top-K
    combined_scores = torch.cat([top_k_scores, similarities], dim=1)
    
    global_chunk_indices = torch.arange(start_idx, end_idx, device=device).unsqueeze(0).expand(len(queries), -1)
    combined_indices = torch.cat([top_k_indices, global_chunk_indices], dim=1)
    
    # Extract new Top 100
    top_k_scores, local_top_k_indices = torch.topk(combined_scores, k=100, dim=1)
    top_k_indices = torch.gather(combined_indices, 1, local_top_k_indices)

minilm_run = {}
top_k_indices_cpu = top_k_indices.cpu().numpy()
for q_idx, qid in enumerate(query_ids):
    minilm_run[qid] = [doc_ids[idx] for idx in top_k_indices_cpu[q_idx]]

save_run_json(minilm_run, "/workspace/results/minilm_run.json")
metrics = evaluate_run(minilm_run, qrels)
print_metrics(metrics, system_name="MiniLM Exact Dense")
"""